In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import pyarrow as pa
import glob

from nonstationary_extremes.GevMCMC import GevMCMC

from scipy.special import logsumexp
from scipy.optimize import minimize

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def log_mean_exp(arr):
    """Compute log(mean(exp(arr))) in a stable way. arr is 1D: log-values a_s"""
    # log(mean(exp(a))) = logsumexp(a) - log(len(a))
    return logsumexp(arr) - np.log(arr.size)

def build_logP_matrix(per_model_fold_logliks, model_names):
    """
    Return logP matrix shape (n_folds, n_models)
    per_model_fold_logliks: dict[model_name] -> list of arrays (one array per fold: loglik_values per posterior draw)
    model_names: list in consistent order
    """
    n_models = len(model_names)
    # assume all models used same folds count
    example = per_model_fold_logliks[model_names[0]]
    n_folds = len(example)
    logP = np.zeros((n_folds, n_models))  # log predictive mass per fold, per model

    for k, m in enumerate(model_names):
        folds = per_model_fold_logliks[m]  # list of arrays, one per fold
        if len(folds) != n_folds:
            raise ValueError(f"Model {m} has {len(folds)} folds but expected {n_folds}")
        for i, loglik_values in enumerate(folds):
            # loglik_values: array of length S (posterior draws), containing log f(y_i | theta_s)
            logP[i, k] = log_mean_exp(np.asarray(loglik_values))
    return logP

def neg_obj_softmax(u):
    """
    u: unconstrained vector length n_models
    weights w = softmax(u)
    returns negative of objective to minimize
    """
    # compute log w_k from u safely
    u = np.asarray(u)
    u = u - np.max(u)  # stability
    log_w = u - logsumexp(u)  # log softmax
    # now for each fold i compute logsumexp(logP[i,:] + log_w)
    term_i = logsumexp(logP + log_w[np.newaxis, :], axis=1)  # shape (n_folds,)
    obj = np.sum(term_i)  # objective to maximize
    return -obj  # minimize negative

def neg_obj_simplex(w):
    # w are weights on simplex already (SLSQP will enforce)
    # avoid zeros: clip
    w = np.maximum(w, 1e-12)
    log_w = np.log(w)
    term_i = logsumexp(logP + log_w[np.newaxis, :], axis=1)
    return -np.sum(term_i)

In [ ]:
n_sims = 10

## Data loading

Here we load in data from previous simulations. Namely:
- Generated data from previous sims
- The true RV delta values from the previous simulations

This allows the use the same data from the information criterion simulation study, so we can have a direct comparison of information criterion to bayesian stacking methods.

In [ ]:
# Read in data from sims
files = sorted(glob.glob("*_SIM_generated_data.parquet"))

dfs = []
sim_offset = 0

for file in files:
    df = pd.read_parquet(file)
    
    max_sim_no = df["Sim_no"].max()
    min_sim_no = df["Sim_no"].min()
    sim_range = max_sim_no - min_sim_no + 1

    df["Sim_no"] = df["Sim_no"] + sim_offset
    
    dfs.append(df)
    
    sim_offset += sim_range

combined = pd.concat(dfs, ignore_index=True)

In [ ]:
files = sorted(glob.glob("*_SIM_results.parquet"))

dfs = []
sim_offset = 0

for file in files:
    df = pd.read_parquet(file)
    
    max_sim_no = df["Sim_no"].max()
    min_sim_no = df["Sim_no"].min()
    sim_range = max_sim_no - min_sim_no + 1

    df["Sim_no"] = df["Sim_no"] + sim_offset
    
    dfs.append(df)
    
    sim_offset += sim_range

sim_results = pd.concat(dfs, ignore_index=True)

In [ ]:
df1_small = sim_results[['Sim_no','True_model','scenario','True_RV_Delta']]

true_rv_wide = (
    df1_small
        .drop_duplicates(subset=['Sim_no','True_model','scenario'])
        .pivot(index=['Sim_no','True_model'],
               columns='scenario',
               values='True_RV_Delta')
        .reset_index()
)

true_rv_wide = true_rv_wide.rename(columns={
    '12': 'True_RV_Delta_12',
    '24': 'True_RV_Delta_24',
    '58': 'True_RV_Delta_58'
})

simulation_results = combined.merge(
    true_rv_wide,
    on=['Sim_no','True_model'],
    how='left'
)

In [ ]:
# If this code has been run before, get the sim_no so that we can iterate through the corresponding datasets for the sim_no
existing_files = sorted(glob.glob("*_CV_model_stacking.parquet"))

if existing_files:
    latest = existing_files[-1]
    existing_df = pd.read_parquet(latest)
    start_sim = existing_df["Sim_no"].max() + 1
else:
    start_sim = 0

n_sims = 30

In [ ]:
runtime = datetime.now().strftime("%Y%m%d_%H%M%S")
CV_FILENAME = f"{runtime}_CV_model_stacking.parquet"

cv_file_exists = os.path.exists(CV_FILENAME)

total_sq_error = 0.0
total_count = 0   # 3 * n_R * n_S

for simulation in range(start_sim, start_sim + n_sims):
    results_tables = []

    per_model_fold_logliks = {}

    for true_model in ["LCC"]:
        
        # Instead of generating new data, we re-use the data from the previous simulation study
        nT = 86
        Tim = np.linspace(0, 1, nT)
        data = simulation_results[(simulation_results['True_model'] == true_model) & (simulation_results['Sim_no'] == simulation)][['12', '24', '58']]

        # Find upper limit of dataset so we can constrain the mcmc
        data_for_constraints = data.iloc[:,:].values.ravel()
        upper_limit = np.ceil(data_for_constraints.min() + 1.2*(data_for_constraints.max()-data_for_constraints.min()))

        # Format data ready for cross validation.
        block_sizes = [9]*6 + [8]*4
        # Equal split for testing
        # block_sizes = [1000]*10
        block_indicies = []

        remaining = list(range(nT))
        for b in block_sizes:
            block = np.random.choice(remaining, size=b, replace=False)
            block_indicies.append(block)
            remaining = [i for i in remaining if i not in block]

        model_scores = {}

        deltaQ_store = {}

        for test_param_setup in ["CCC", "LCC", "LLC", "QCC", "ACC", "LLL", "QLC", "QLL", "QQC", "QQL", "QQQ"]:

            print(f" Tesing model {test_param_setup}")

            total_lpred = 0
            lpred = []
            all_loglik = []

            for i, block in enumerate(block_indicies):
                print(f" Fold {i+1}/10")

                train_idx = np.setdiff1d(np.arange(nT), block)
                test_idx = block

                train_data = data.iloc[train_idx]
                test_data = data.iloc[test_idx]
                
                Tim_train = Tim[train_idx]
                Tim_test = Tim[test_idx]

                gev_mcmc = GevMCMC(train_data, test_param_setup, upper_limit, verbose=False)

                n_samples = 10000
                n2plt = 5000
                burn_in = 1000
                thinning = 20
                beta = 0.05
                NGTSTR = 0.1

                samples, total_accepted, ar, nloglikelihood, t_nloglikelihood = gev_mcmc.run(n_samples, n2plt, burn_in, thinning, beta, NGTSTR)

                nll_pred, log_pred, loglik_values = gev_mcmc.posterior_predictive_nll(
                    test_data, samples.to_numpy(dtype=float), time_steps=Tim_test
                )

                all_loglik.append(loglik_values)
            
            deltaQ_samples = gev_mcmc.plot_return_values(samples)
            deltaQ_store[test_param_setup] = deltaQ_samples

            all_loglik = np.array(all_loglik)
            sum_loglik = np.sum(all_loglik, axis=0)
            per_model_fold_logliks[test_param_setup] = all_loglik

            # Two metrics:
            # mean
            avg = np.mean(sum_loglik)

            # exp -> mean -> log
            exp_arr = np.exp(sum_loglik)
            mean_exp = np.mean(exp_arr)
            log_mean_exp_array = np.log(mean_exp)

            model_scores[test_param_setup] = {
                "avg": avg,
                "log_mean_exp_array": log_mean_exp_array
            }
        
        ## Model Stacking
        model_names = list(per_model_fold_logliks.keys())  # keep order consistent
        logP = build_logP_matrix(per_model_fold_logliks, model_names)  # shape (n_folds, n_models)

        n_models = len(model_names)

        # initial guess: uniform
        w0 = np.ones(n_models) / n_models

        # we can also add a small regulariser to encourage interior solution (optional)
        res = minimize(neg_obj_softmax, x0=np.zeros(n_models), method='BFGS')

        if not res.success:
            # fallback to SLSQP with constraints on simplex
            cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
            bounds = [(0.0, 1.0)] * n_models
            w0 = np.ones(n_models) / n_models
            res2 = minimize(neg_obj_simplex, w0, method='SLSQP', bounds=bounds, constraints=cons)
            if not res2.success:
                raise RuntimeError("Stacking optimization failed: " + res2.message)
            weights = res2.x
        else:
            # convert optimised u to weights
            u_opt = res.x
            weights = np.exp(u_opt - logsumexp(u_opt))

        # Package results
        stacking_weights = dict(zip(model_names, weights))

        # Refit models on FULL data
        deltaQ_store = {}

        for model_name in model_names:

            print(f"Refitting full model {model_name}")

            gev_mcmc_full = GevMCMC(
                data,
                model_name,
                upper_limit,
                verbose=False
            )

            samples_full, *_ = gev_mcmc_full.run(
                n_samples, n2plt,
                burn_in, thinning,
                beta, NGTSTR
            )

            # Compute ΔQ posterior samples from FULL posterior
            deltaQ_samples = np.asarray(
                gev_mcmc_full.plot_return_values(samples_full)
            )

            # Ensure shape is (3, n_samples)
            deltaQ_store[model_name] = deltaQ_samples
                
        # Compute BMA posterior draws
        # shape (3, n_samples)
        bma_draws = None

        for m in model_names:
            w = stacking_weights[m]
            model_draws = deltaQ_store[m]  # (3, n_samples)

            if bma_draws is None:
                bma_draws = w * model_draws
            else:
                bma_draws += w * model_draws
        
        # ===== Accumulate RMSE terms =====
        # True_DeltaQ must be shape (3,)
        true_rv_delta = simulation_results[(simulation_results['Sim_no'] == simulation) & (simulation_results['True_model'] == true_model)]

        true_rv_delta = true_rv_delta[
            ['True_RV_Delta_12',
            'True_RV_Delta_24',
            'True_RV_Delta_58']
        ].iloc[0].to_numpy()

        sq_error = (true_rv_delta - bma_draws) ** 2
        
        sim_sq_error = np.sum(sq_error) # Sum over models & samples
        sim_count = sq_error.size   # 3 * n_samples i.e 750

        model_names = list(model_scores.keys())

        for model_name in model_names:
            results_tables.append({
                "Sim_no": simulation,
                "True_model": true_model,
                "model": model_name,
                "avg": model_scores[model_name]["avg"],
                "log_mean_exp_array": model_scores[model_name]["log_mean_exp_array"],
                "stack_weight": stacking_weights[model_name],
                "sim_sq_error": sim_sq_error,
                "sim_count": sim_count
            })  
        
    cv_model_probabilities = pd.DataFrame(results_tables)
    
    cv_table = pa.Table.from_pandas(cv_model_probabilities)

    if cv_file_exists:
        existing_table = pq.read_table(CV_FILENAME)
        cv_table = pa.concat_tables([existing_table, cv_table])

    pq.write_table(
        cv_table,
        CV_FILENAME,
        compression="snappy"
    )

    cv_file_exists = True

## Results

Here we load the results from the simulations above. Then calculate the RMSE & 2.5, 97.5 percentiles for the stacking weights

In [ ]:
files = sorted(glob.glob("*_CV_model_stacking.parquet"))

dfs = []
sim_offset = 0

for file in files:
    df = pd.read_parquet(file)
    
    # max_sim_no = df["Sim_no"].max()
    # min_sim_no = df["Sim_no"].min()
    # sim_range = max_sim_no - min_sim_no + 1

    # df["Sim_no"] = df["Sim_no"] + sim_offset
    
    dfs.append(df)
    
    sim_offset += sim_range

combined_stacking = pd.concat(dfs, ignore_index=True)

In [ ]:
# Calculate overall RMSE for bayesian stacking
total_sq_error = combined_stacking["sim_sq_error"].sum()
total_count = combined_stacking["sim_count"].sum()

rmse = np.sqrt(total_sq_error / total_count)
rmse

np.float64(3.9880822217956564)

In [ ]:
stacking_summary = (
    combined_stacking
    .groupby(["True_model", "model"])["stack_weight"]
    .agg(
        mean="mean",
        lower=lambda x: x.quantile(0.025),
        upper=lambda x: x.quantile(0.975)
    )
    .reset_index()
    .round(2)
)

stacking_summary

,True_model,model,mean,lower,upper
0,LCC,ACC,0.18,0.0,0.91
1,LCC,CCC,0.14,0.0,0.59
2,LCC,LCC,0.40,0.0,1.00
3,LCC,LLC,0.07,0.0,0.74
4,LCC,LLL,0.04,0.0,0.46
5,LCC,QCC,0.09,0.0,0.79
6,LCC,QLC,0.01,0.0,0.21
7,LCC,QLL,0.01,0.0,0.00
8,LCC,QQC,0.01,0.0,0.12
9,LCC,QQL,0.01,0.0,0.12
